In [ ]:
import numpy as np
import pandas as pd

In [ ]:
def sigmoid(x):
    x = np.asarray(x, dtype=np.float64)
    x = np.clip(x, -500, 500)
    return 1 / (1 + np.exp(-x))

In [ ]:



class LogisticRegression:

    def __init__(self, lr=0.001, n_iters=1000):
        self.lr = lr
        self.n_iters = n_iters
        self.weights = None
        self.bias = None

  
    def fit(self, X, y):
     X = np.asarray(X, dtype=np.float64)
     y = np.asarray(y, dtype=np.float64).reshape(-1)

     n_samples, n_features = X.shape

     self.weights = np.zeros(n_features, dtype=np.float64)
     self.bias = 0.0

     for i in range(self.n_iters):
        linear_pred = np.dot(X, self.weights) + self.bias
        predictions = sigmoid(linear_pred)

        dw = np.dot(X.T, predictions - y) / n_samples
        db = np.sum(predictions - y) / n_samples

        self.weights -= self.lr * dw
        self.bias -= self.lr * db

        if i % 50 == 0:
            print("Iteration:", i)


    def predict(self, X):
        linear_pred = np.dot(X, self.weights) + self.bias
        y_pred = sigmoid(linear_pred)

        class_pred = [0 if y <= 0.5 else 1 for y in y_pred]
        return np.array(class_pred)



    

In [ ]:


train_data = pd.read_csv("crime_train_cleaned.csv")
test_data = pd.read_csv("crime_cleaned.csv")

target = "closed"



In [ ]:

def clean_target(series):
    return (
        series.astype(str)
        .str.strip()
        .str.lower()
        .map({"no": 0, "yes": 1, "0": 0, "1": 1})
    )

y_train_series = clean_target(train_data[target])
y_test_series = clean_target(test_data[target])

# Remove rows with invalid or missing target labels
train_valid = y_train_series.notna()
test_valid = y_test_series.notna()

train_data = train_data.loc[train_valid].copy()
test_data = test_data.loc[test_valid].copy()

y_train = y_train_series.loc[train_valid].to_numpy(dtype=int)
y_test = y_test_series.loc[test_valid].to_numpy(dtype=int)

In [ ]:
X_train = train_data.drop(columns=[target]).copy()
X_test = test_data.drop(columns=[target]).copy()

# Drop ID-like columns that do not provide useful predictive information
id_cols = [
    col for col in X_train.columns
    if col.strip().lower() in ["unnamed: 0", "id", "index"]
]

X_train.drop(columns=id_cols, inplace=True, errors="ignore")
X_test.drop(columns=id_cols, inplace=True, errors="ignore")

# Keep the same feature columns in both datasets
X_test = X_test.reindex(columns=X_train.columns)


In [ ]:
for col in X_train.columns:
    if (
        pd.api.types.is_object_dtype(X_train[col])
        and X_train[col].astype(str).str.contains(
            r"\d{1,2}-\d{1,2}-\d{4}\s+\d{1,2}:\d{2}",
            regex=True
        ).any()
    ):
        train_date = pd.to_datetime(
            X_train[col], format="%d-%m-%Y %H:%M", errors="coerce"
        )
        test_date = pd.to_datetime(
            X_test[col], format="%d-%m-%Y %H:%M", errors="coerce"
        )

        for name, date_values in [
            ("year", (train_date.dt.year, test_date.dt.year)),
            ("month", (train_date.dt.month, test_date.dt.month)),
            ("day", (train_date.dt.day, test_date.dt.day)),
            ("hour", (train_date.dt.hour, test_date.dt.hour)),
        ]:
            X_train[col + "_" + name] = date_values[0]
            X_test[col + "_" + name] = date_values[1]

        X_train.drop(columns=[col], inplace=True)
        X_test.drop(columns=[col], inplace=True)

In [ ]:
combined = pd.concat([X_train, X_test], axis=0)

combined = pd.get_dummies(combined, dummy_na=True)

X_train = combined.iloc[:len(X_train)].copy()
X_test = combined.iloc[len(X_train):].copy()

In [ ]:
X_train = X_train.to_numpy(dtype=float)
X_test = X_test.to_numpy(dtype=float)

# Standardize features using training statistics
mean = X_train.mean(axis=0)
std = X_train.std(axis=0)
std[std == 0] = 1

X_train = (X_train - mean) / std
X_test = (X_test - mean) / std

In [ ]:
model = LogisticRegression(lr=0.01, n_iters=2000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

correct = np.sum(y_pred == y_test)
total = len(y_test)

accuracy = correct / total if total > 0 else 0

print("Correct predictions:", correct)
print("Total predictions:", total)
print(f"Accuracy: {accuracy:.4f}")
print(f"Accuracy percentage: {accuracy * 100:.2f}%")

In [ ]:
TP = np.sum((y_test == 1) & (y_pred == 1))
TN = np.sum((y_test == 0) & (y_pred == 0))
FP = np.sum((y_test == 0) & (y_pred == 1))
FN = np.sum((y_test == 1) & (y_pred == 0))

print("\nConfusion Matrix")
print("                 Predicted 0  Predicted 1")
print(f"Actual 0         {TN:10d}  {FP:10d}")
print(f"Actual 1         {FN:10d}  {TP:10d}")

precision = TP / (TP + FP) if TP + FP else 0
recall = TP / (TP + FN) if TP + FN else 0
f1 = (
    2 * precision * recall / (precision + recall)
    if precision + recall else 0
)

print(f"\nPrecision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1:.4f}")